# NER Archelec — Fine-tuning CamemBERT
## Comparaison CamemBERT-base vs CamemBERT-NER

**Corpus** : Professions de foi électorales françaises (1973/1978)  
**Tâche** : Named Entity Recognition (PER, ORG, LOC, MISC)  
**Baseline** : spaCy fr_core_news_lg → F1 = 6.44%

---

### Pipeline
```
train.json (6936 docs) ──┐
val.json   (867 docs)  ──┼──► Trainer HuggingFace ──► Évaluation sur test.json
test.json  (867 docs)  ──┘
```

### Labels
| ID | Label  | Description |
|----|--------|-------------|
| 0  | O      | Hors entité |
| 1  | B-PER  | Début nom candidat |
| 2  | I-PER  | Suite nom candidat |
| 3  | B-ORG  | Début parti politique |
| 4  | I-ORG  | Suite parti politique |
| 5  | B-LOC  | Début département |
| 6  | I-LOC  | Suite département |
| 7  | B-MISC | Début profession |
| 8  | I-MISC | Suite profession |

In [1]:
# ── Installation des dépendances ──────────────────────────────────────────────
!pip install transformers datasets seaborn -q
!pip install accelerate -q

import os, json, shutil
import numpy as np
import torch
from pathlib import Path
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback
)
from datasets import Dataset, DatasetDict

print(f"PyTorch    : {torch.__version__}")
print(f"GPU dispo  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU        : {torch.cuda.get_device_name(0)}")
    print(f"VRAM       : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch    : 2.10.0+cpu
GPU dispo  : False


In [3]:
# ── Vérification des fichiers dans /content/ ──────────────────────────────────
# Upload les 4 fichiers via l explorateur de fichiers VS Code (glisse dans /content/)
# -> train.json, val.json, test.json, label2id.json

RESULTS_DIR = "/content/"

fichiers = ["train.json", "val.json", "test.json", "label2id.json"]
for f in fichiers:
    chemin = f"/content/{f}"
    if os.path.exists(chemin):
        taille = os.path.getsize(chemin) / 1024 / 1024
        print(f"OK {f} ({taille:.1f} MB)")
    else:
        print(f"MANQUANT {f} -- uploade-le dans /content/ via l explorateur de fichiers")

In [5]:
# ── Chargement des données ────────────────────────────────────────────────────

# Charger label mapping
with open("/content/label2id.json") as f:
    LABEL2ID = json.load(f)
ID2LABEL  = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = len(LABEL2ID)
print(f"Labels ({NUM_LABELS}) : {LABEL2ID}")

# Charger les splits
def charger_split(chemin):
    with open(chemin, encoding="utf-8") as f:
        data = json.load(f)
    return Dataset.from_list([
        {
            "input_ids": d["input_ids"],
            "ner_tags":  d["ner_tags"],
            "id":        d["id"],
            "annee":     d["annee"]
        }
        for d in data
    ])

dataset = DatasetDict({
    "train": charger_split("/content/train.json"),
    "val":   charger_split("/content/val.json"),
    "test":  charger_split("/content/test.json"),
})

print(f"\nTrain : {len(dataset['train'])} docs")
print(f"Val   : {len(dataset['val'])} docs")
print(f"Test  : {len(dataset['test'])} docs")

# Vérifier un exemple
ex = dataset["train"][0]
print(f"\nExemple :")
print(f"  input_ids (5 premiers) : {ex['input_ids'][:5]}")
print(f"  ner_tags  (5 premiers) : {ex['ner_tags'][:5]}")
print(f"  longueur               : {len(ex['input_ids'])} tokens")

In [ ]:
# ── Preprocessing ─────────────────────────────────────────────────────────────

MAX_LENGTH  = 512
PAD_TOKEN_ID = 1  # <pad> pour CamemBERT

def preprocess_batch(examples):
    batch_input_ids      = []
    batch_attention_mask = []
    batch_labels         = []

    for input_ids, ner_tags in zip(examples["input_ids"], examples["ner_tags"]):
        # Tronquer
        input_ids = input_ids[:MAX_LENGTH]
        ner_tags  = ner_tags[:MAX_LENGTH]

        # Padding
        pad_len        = MAX_LENGTH - len(input_ids)
        attention_mask = [1] * len(input_ids) + [0] * pad_len
        input_ids      = input_ids + [PAD_TOKEN_ID] * pad_len
        labels         = ner_tags  + [-100] * pad_len

        batch_input_ids.append(input_ids)
        batch_attention_mask.append(attention_mask)
        batch_labels.append(labels)

    return {
        "input_ids":      batch_input_ids,
        "attention_mask": batch_attention_mask,
        "labels":         batch_labels,
    }

dataset_proc = dataset.map(
    preprocess_batch,
    batched=True,
    remove_columns=["ner_tags", "id", "annee"]
)
dataset_proc.set_format("torch")
print("✅ Dataset préparé pour le Trainer")
print(f"   Features : {dataset_proc['train'].features}")

In [ ]:
# ── Fonction de métriques ─────────────────────────────────────────────────────

def extraire_spans(tags):
    """
    Extrait les spans (type, debut, fin) depuis une séquence BIO.
    Accepte des entiers (indices) ou des chaînes ('B-PER', etc.).
    """
    spans = set()
    i = 0
    while i < len(tags):
        tag = tags[i] if isinstance(tags[i], str) else ID2LABEL[tags[i]]
        if tag.startswith("B-"):
            entite = tag[2:]
            debut  = i
            i += 1
            while i < len(tags):
                t = tags[i] if isinstance(tags[i], str) else ID2LABEL[tags[i]]
                if t == f"I-{entite}":
                    i += 1
                else:
                    break
            spans.add((entite, debut, i))
        else:
            i += 1
    return spans


def compute_metrics(pred):
    """
    Calcule Précision, Rappel et F1 par type d'entité et globalement.
    Évaluation au niveau des spans (entité complète : type + positions).
    """
    predictions, labels = pred
    predictions = np.argmax(predictions, axis=2)

    stats        = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})
    stats_global = {"tp": 0, "fp": 0, "fn": 0}

    for pred_seq, label_seq in zip(predictions, labels):
        # Ignorer les positions paddées (label == -100)
        pred_clean  = [ID2LABEL[p] for p, l in zip(pred_seq, label_seq) if l != -100]
        label_clean = [ID2LABEL[l] for l in label_seq if l != -100]

        spans_pred = extraire_spans(pred_clean)
        spans_gold = extraire_spans(label_clean)

        for span in spans_pred:
            if span in spans_gold:
                stats[span[0]]["tp"] += 1
                stats_global["tp"]   += 1
            else:
                stats[span[0]]["fp"] += 1
                stats_global["fp"]   += 1
        for span in spans_gold:
            if span not in spans_pred:
                stats[span[0]]["fn"] += 1
                stats_global["fn"]   += 1

    def prf(tp, fp, fn):
        p  = tp / (tp + fp) if (tp + fp) > 0 else 0
        r  = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2*p*r / (p+r)  if (p  + r) > 0 else 0
        return p*100, r*100, f1*100

    res = {}
    for entite in ["PER", "ORG", "LOC", "MISC"]:
        s = stats[entite]
        p, r, f1 = prf(s["tp"], s["fp"], s["fn"])
        res[f"f1_{entite}"]        = round(f1, 2)
        res[f"precision_{entite}"] = round(p,  2)
        res[f"recall_{entite}"]    = round(r,  2)

    p_g, r_g, f1_g = prf(
        stats_global["tp"], stats_global["fp"], stats_global["fn"]
    )
    res["f1_global"]        = round(f1_g, 2)
    res["precision_global"] = round(p_g,  2)
    res["recall_global"]    = round(r_g,  2)
    return res


print("✅ Fonction compute_metrics définie")

## Modèle 1 — CamemBERT-base

Modèle BERT français généraliste, fine-tuné sur notre corpus NER Archelec.

| Hyperparamètre | Valeur |
|----------------|--------|
| Learning rate | 2e-5 |
| Epochs | 5 (early stopping patience=2) |
| Batch size train | 16 |
| Batch size eval | 32 |
| Weight decay | 0.01 |
| Warmup ratio | 10% |
| Mixed precision | fp16 |

In [ ]:
# ── Entraînement CamemBERT-base ───────────────────────────────────────────────

MODEL_1_NAME = "camembert-base"
print(f"Chargement {MODEL_1_NAME}...")

model_1 = AutoModelForTokenClassification.from_pretrained(
    MODEL_1_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True
)

args_1 = TrainingArguments(
    output_dir                  = "/content/camembert_base_ner",
    num_train_epochs            = 5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    learning_rate               = 2e-5,
    weight_decay                = 0.01,
    warmup_ratio                = 0.1,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1_global",
    greater_is_better           = True,
    logging_steps               = 100,
    fp16                        = True,
    report_to                   = "none",
    save_total_limit            = 1,
)

trainer_1 = Trainer(
    model           = model_1,
    args            = args_1,
    train_dataset   = dataset_proc["train"],
    eval_dataset    = dataset_proc["val"],
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)]
)

print("🚀 Entraînement CamemBERT-base...")
trainer_1.train()
print("✅ Entraînement terminé !")

In [ ]:
print("=== EVALUATION CamemBERT-base sur TEST ===")
results_1 = trainer_1.evaluate(dataset_proc["test"])

print(f"{'Entité':<10} {'Precision':>12} {'Recall':>10} {'F1':>10}")
print("-" * 45)

for entite in ["PER", "ORG", "LOC", "MISC"]:
    p = results_1.get(f"eval_precision_{entite}", 0)
    r = results_1.get(f"eval_recall_{entite}", 0)
    f1 = results_1.get(f"eval_f1_{entite}", 0)
    print(f"{entite:<10} {p:>11.2f}% {r:>9.2f}% {f1:>9.2f}%")

print("-" * 45)

p_g = results_1.get("eval_precision_global", 0)
r_g = results_1.get("eval_recall_global", 0)
f1_g = results_1.get("eval_f1_global", 0)

print(f"{'GLOBAL':<10} {p_g:>11.2f}% {r_g:>9.2f}% {f1_g:>9.2f}%")

# Sauvegarder les résultats dans /content/
trainer_1.save_model("/content/camembert_base_ner_model")

with open("/content/results_camembert_base.json", "w") as f:
    json.dump(results_1, f, indent=2)

print("OK Modèle sauvegardé dans /content/camembert_base_ner_model")

## Modèle 2 — CamemBERT-NER (`Jean-Baptiste/camembert-ner`)

Modèle CamemBERT **déjà fine-tuné** sur des données NER françaises (WikiNER, etc.).  
On l'adapte par transfer learning sur notre corpus Archelec.

| Hyperparamètre | Valeur | Justification |
|----------------|--------|---------------|
| Learning rate | **1e-5** | Plus faible — modèle déjà spécialisé NER |
| Epochs | 5 (early stopping) | Même protocole |
| Batch size | 16 / 32 | Idem |
| `ignore_mismatched_sizes` | True | Adapter la tête de classification à nos 9 labels |

In [ ]:
# # ── Libérer la mémoire GPU avant le 2e modèle ─────────────────────────────────
# del model_1
# torch.cuda.empty_cache()
# print("✅ Mémoire GPU libérée")

# # ── Entraînement CamemBERT-NER ────────────────────────────────────────────────
# MODEL_2_NAME = "Jean-Baptiste/camembert-ner"
# print(f"\nChargement {MODEL_2_NAME}...")

# model_2 = AutoModelForTokenClassification.from_pretrained(
#     MODEL_2_NAME,
#     num_labels=NUM_LABELS,
#     id2label=ID2LABEL,
#     label2id=LABEL2ID,
#     ignore_mismatched_sizes=True
# )

# args_2 = TrainingArguments(
#     output_dir                  = "/content/camembert_ner_finetuned",
#     num_train_epochs            = 5,
#     per_device_train_batch_size = 16,
#     per_device_eval_batch_size  = 32,
#     learning_rate               = 1e-5,
#     weight_decay                = 0.01,
#     warmup_ratio                = 0.1,
#     eval_strategy               = "epoch",
#     save_strategy               = "epoch",
#     load_best_model_at_end      = True,
#     metric_for_best_model       = "f1_global",
#     greater_is_better           = True,
#     logging_steps               = 100,
#     fp16                        = True,
#     report_to                   = "none",
#     save_total_limit            = 1,
# )

# trainer_2 = Trainer(
#     model           = model_2,
#     args            = args_2,
#     train_dataset   = dataset_proc["train"],
#     eval_dataset    = dataset_proc["val"],
#     compute_metrics = compute_metrics,
#     callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)]
# )

# print("🚀 Entraînement CamemBERT-NER...")
# trainer_2.train()
# print("✅ Entraînement terminé !")

In [ ]:
from collections import OrderedDict

del model_2
torch.cuda.empty_cache()
print("Mémoire GPU libérée")

print("Chargement et correction des poids Jean-Baptiste/camembert-ner...")
from transformers import CamembertForTokenClassification

raw_model = CamembertForTokenClassification.from_pretrained(
    "Jean-Baptiste/camembert-ner",
    ignore_mismatched_sizes=True
)
old_sd = raw_model.state_dict()

# Renommer beta->weight, gamma->bias ET exclure le classifier (taille incompatible)
fixed_sd = OrderedDict()
for k, v in old_sd.items():
    if k.startswith("classifier"):   # ← exclure la tête de classification
        continue
    new_k = k.replace(".gamma", ".weight").replace(".beta", ".bias")
    fixed_sd[new_k] = v

print(f"Clés chargées depuis camembert-ner : {len(fixed_sd)}")

# Modèle vierge camembert-base avec nos 9 labels
model_2 = AutoModelForTokenClassification.from_pretrained(
    "camembert-base",
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True
)

# Injecter les poids encodeur corrigés (classifier reste aléatoire = normal)
missing, unexpected = model_2.load_state_dict(fixed_sd, strict=False)
print(f"Clés manquantes  : {len(missing)}  (classifier → OK, réinitialisé)")
print(f"Clés inattendues : {len(unexpected)}")
print("Poids encodeur chargés correctement !")

args_2 = TrainingArguments(
    output_dir                  = "/content/camembert_ner_finetuned",
    num_train_epochs            = 5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    learning_rate               = 2e-5,
    weight_decay                = 0.01,
    warmup_ratio                = 0.1,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1_global",
    greater_is_better           = True,
    logging_steps               = 100,
    fp16                        = True,
    report_to                   = "none",
    save_total_limit            = 1,
)

trainer_2 = Trainer(
    model           = model_2,
    args            = args_2,
    train_dataset   = dataset_proc["train"],
    eval_dataset    = dataset_proc["val"],
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=3)]
)

print("Entraînement CamemBERT-NER corrigé...")
trainer_2.train()
print("Entraînement terminé !")

   

In [ ]:
# del model_1
# torch.cuda.empty_cache()
# print("Mémoire GPU libérée")

# MODEL_2_NAME = "Jean-Baptiste/camembert-ner"
# print(f"\nChargement {MODEL_2_NAME}...")

# model_2 = AutoModelForTokenClassification.from_pretrained(
#     MODEL_2_NAME,
#     num_labels=NUM_LABELS,
#     id2label=ID2LABEL,
#     label2id=LABEL2ID,
#     ignore_mismatched_sizes=True
# )

# args_2 = TrainingArguments(
#     output_dir                  = "/content/camembert_ner_finetuned",
#     num_train_epochs            = 5,
#     per_device_train_batch_size = 16,
#     per_device_eval_batch_size  = 32,
#     learning_rate               = 2e-5,   # augmenté
#     weight_decay                = 0.01,
#     warmup_ratio                = 0.1,
#     eval_strategy               = "epoch",
#     save_strategy               = "epoch",
#     load_best_model_at_end      = True,
#     metric_for_best_model       = "f1_global",
#     greater_is_better           = True,
#     logging_steps               = 100,
#     fp16                        = True,
#     report_to                   = "none",
#     save_total_limit            = 1,
# )

# trainer_2 = Trainer(
#     model           = model_2,
#     args            = args_2,
#     train_dataset   = dataset_proc["train"],
#     eval_dataset    = dataset_proc["val"],
#     compute_metrics = compute_metrics,
#     callbacks       = [EarlyStoppingCallback(early_stopping_patience=3)]  # augmenté
# )

# print("Entraînement CamemBERT-NER...")
# trainer_2.train()
# print("Entraînement terminé !")


In [ ]:
# ── Évaluation CamemBERT-NER sur le test set ──────────────────────────────────

print("=== ÉVALUATION CamemBERT-NER sur TEST ===")
results_2 = trainer_2.evaluate(dataset_proc["test"])

print(f"{'Entité':<10} {'Precision':>12} {'Recall':>10} {'F1':>10}")
print("-" * 45)

for entite in ["PER", "ORG", "LOC", "MISC"]:
    p = results_2.get(f"eval_precision_{entite}", 0)
    r = results_2.get(f"eval_recall_{entite}", 0)
    f1 = results_2.get(f"eval_f1_{entite}", 0)
    print(f"{entite:<10} {p:>11.2f}% {r:>9.2f}% {f1:>9.2f}%")

print("-" * 45)

p_g = results_2.get("eval_precision_global", 0)
r_g = results_2.get("eval_recall_global", 0)
f1_g = results_2.get("eval_f1_global", 0)

print(f"{'GLOBAL':<10} {p_g:>11.2f}% {r_g:>9.2f}% {f1_g:>9.2f}%")

# Sauvegarder les résultats dans /content/
trainer_2.save_model("/content/camembert_ner_archelec_model")

with open("/content/results_camembert_ner.json", "w") as f:
    json.dump(results_2, f, indent=2)

print("OK Modèle sauvegardé dans /content/camembert_ner_archelec_model")

## Résultats comparatifs

Comparaison des 3 approches :
1. **spaCy baseline** — modèle généraliste, sans entraînement sur nos données
2. **CamemBERT-base** — BERT français fine-tuné from scratch sur Archelec
3. **CamemBERT-NER** — BERT déjà spécialisé NER, adapté sur Archelec

In [ ]:
# ── Tableau comparatif final ──────────────────────────────────────────────────

BASELINE = {
    "eval_f1_PER":        11.68, "eval_precision_PER":  6.50, "eval_recall_PER":  57.1,
    "eval_f1_ORG":         8.52, "eval_precision_ORG":  4.66, "eval_recall_ORG":  49.1,
    "eval_f1_LOC":         5.63, "eval_precision_LOC":  3.10, "eval_recall_LOC":  30.4,
    "eval_f1_MISC":        0.15, "eval_precision_MISC": 0.08, "eval_recall_MISC":  2.2,
    "eval_f1_global":      6.44,
    "eval_precision_global": 3.49, "eval_recall_global": 41.3
}

print("\n" + "=" * 80)
print("TABLEAU COMPARATIF FINAL — 3 MODÈLES NER ARCHELEC")
print("=" * 80)
print(f"{'Modèle':<28} {'F1-PER':>8} {'F1-ORG':>8} {'F1-LOC':>8} {'F1-MISC':>9} {'F1-Global':>11}")
print("-" * 80)

modeles = [
    ("spaCy baseline",  BASELINE),
    ("CamemBERT-base",  results_1),
    ("CamemBERT-NER",   results_2),
]

for nom, res in modeles:
    def g(key): return res.get(f"eval_{key}", res.get(key, 0))
    print(f"{nom:<28} {g('f1_PER'):>7.1f}% {g('f1_ORG'):>7.1f}% "
          f"{g('f1_LOC'):>7.1f}% {g('f1_MISC'):>8.1f}% {g('f1_global'):>10.1f}%")

print("=" * 80)

# Gain par rapport à la baseline
print("\nGain par rapport à la baseline spaCy :")
for nom, res in [("CamemBERT-base", results_1), ("CamemBERT-NER", results_2)]:
    f1_model = res.get("eval_f1_global", 0)
    gain = f1_model - BASELINE["eval_f1_global"]
    print(f"  {nom:<22} : +{gain:.1f} points F1 ({f1_model:.1f}% vs {BASELINE['eval_f1_global']}%)")

In [ ]:
# ── Graphiques comparatifs ────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

entites  = ["PER", "ORG", "LOC", "MISC"]
noms     = ["spaCy\nbaseline", "CamemBERT\nbase", "CamemBERT\nNER"]
couleurs = ["#ef4444", "#4a9eed", "#22c55e"]
x        = np.arange(len(entites))
width    = 0.25

# Graphique 1 — F1 par entité
for i, (nom, res, couleur) in enumerate(zip(noms, [BASELINE, results_1, results_2], couleurs)):
    def g(e): return res.get(f"eval_f1_{e}", res.get(f"f1_{e}", 0))
    scores = [g("PER"), g("ORG"), g("LOC"), g("MISC")]
    axes[0].bar(x + i*width, scores, width,
                label=nom.replace("\n", " "), color=couleur, alpha=0.85)

axes[0].set_xlabel("Type d'entité", fontsize=12)
axes[0].set_ylabel("F1 Score (%)", fontsize=12)
axes[0].set_title("F1 par entité — Comparaison des 3 modèles", fontsize=13)
axes[0].set_xticks(x + width)
axes[0].set_xticklabels(entites, fontsize=12)
axes[0].legend(fontsize=11)
axes[0].set_ylim(0, 100)
axes[0].grid(axis="y", alpha=0.3)

# Graphique 2 — F1 global
f1_globaux = [
    BASELINE["eval_f1_global"],
    results_1.get("eval_f1_global", 0),
    results_2.get("eval_f1_global", 0),
]
bars = axes[1].bar(
    ["spaCy\nbaseline", "CamemBERT\nbase", "CamemBERT\nNER"],
    f1_globaux, color=couleurs, alpha=0.85, width=0.5
)
for bar, val in zip(bars, f1_globaux):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f"{val:.1f}%", ha="center", fontsize=13, fontweight="bold"
    )

axes[1].set_ylabel("F1 Global (%)", fontsize=12)
axes[1].set_title("F1 Global — Comparaison des 3 modèles", fontsize=13)
axes[1].set_ylim(0, 100)
axes[1].grid(axis="y", alpha=0.3)

plt.suptitle(
    "NER Archelec — Résultats comparatifs\n(Corpus : professions de foi 1973/1978)",
    fontsize=14, fontweight="bold"
)
plt.tight_layout()
plt.savefig("/content/comparaison_modeles.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Graphique sauvegardé sur Drive")

## Analyse des erreurs — Exemples de prédictions

Inspecter les prédictions sur quelques documents du test set pour comprendre les erreurs résiduelles.

In [ ]:
# ── Analyse qualitative — exemples de prédictions ────────────────────────────
# Utilise le meilleur des deux modèles (CamemBERT-NER)

model_2.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_2.to(device)

# Recharger le test set original (avec les tokens lisibles)
with open("/content/test.json", encoding="utf-8") as f:
    test_raw = json.load(f)

print("=== EXEMPLES DE PRÉDICTIONS (CamemBERT-NER) ===")

for doc in test_raw[:3]:
    input_ids = torch.tensor([doc["input_ids"][:512]], dtype=torch.long).to(device)
    attention_mask = torch.ones_like(input_ids).to(device)

    with torch.no_grad():
        logits = model_2(input_ids=input_ids, attention_mask=attention_mask).logits

    preds = torch.argmax(logits, dim=-1)[0].cpu().tolist()
    tokens = doc["tokens"][:512]
    gold   = [ID2LABEL[t] for t in doc["ner_tags"][:512]]
    pred_labels = [ID2LABEL[p] for p in preds]

    print(f"\n--- ID: {doc['id']} ({doc['annee']}) ---")
    print(f"{'Token':<20} {'Gold':>10} {'Pred':>10} {'Match':>8}")
    print("-" * 52)

    # Afficher seulement les tokens avec une entité (gold ou pred)
    entites_affichees = 0
    for token, g, p in zip(tokens, gold, pred_labels):
        if g != "O" or p != "O":
            match = "✅" if g == p else "❌"
            token_clean = token.replace("▁", " ").strip()
            print(f"  {token_clean:<18} {g:>10} {p:>10} {match:>8}")
            entites_affichees += 1
            if entites_affichees >= 15:
                print("  [... tronqué ...]")
                break

    if entites_affichees == 0:
        print("  (aucune entité dans ce document)")

## Résumé et conclusions

### Ce que nous avons fait

1. **Préparation des données** (scripts 01–05 en local) :
   - 8 670 professions de foi annotées par distant supervision
   - 4 types d'entités : PER, ORG, LOC, MISC
   - Split 80/10/10 stratifié par année

2. **Baseline spaCy** (script 06) :
   - F1 global = **6.44%** — modèle généraliste, non adapté au domaine

3. **Fine-tuning CamemBERT** (ce notebook) :
   - CamemBERT-base : fine-tuning from scratch
   - CamemBERT-NER  : transfer learning depuis un modèle déjà spécialisé

### Limites et perspectives

- Les annotations sont automatiques (distant supervision) → bruit possible
- MISC (profession) est la classe la plus difficile : absente du CSV dans ~36% des cas
- Une annotation manuelle sur un sous-ensemble améliorerait significativement les scores
- Extension possible : inclure les communes (LOC supplémentaire) et les années 1968, 1981